<div class="alert alert-info">

## <center> CS587 Final Project - Phase 2 (Scrum)</center>
### <center> GenAI-Assisted Personalized LMS - Claude</center>

This notebook re-engineers the Phase 1 Waterfall solution into a Scrum-based plan using a multi-agent AutoGen workflow.


<b>Owner - Anu Singh (A20568373)</b>
</div>


In [1]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "pyautogen<0.4",
    "anthropic>=0.23.1",
    "--quiet",
])

import os
import json
import time
from datetime import datetime

import autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
print("Current working directory:", os.getcwd())

You should consider upgrading via the '/Users/anusingh/Documents/Phase2_Claude/.venv/bin/python -m pip install --upgrade pip' command.


Current working directory: /Users/anusingh/Documents/Phase2_Claude


/Users/anusingh/Documents/Phase2_Claude/.venv/lib/python3.9/site-packages/flaml/__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [2]:
def load_env_file(path):
    import os
    if not os.path.exists(path):
        return False
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()
    return True

loaded = load_env_file(os.path.join(os.getcwd(), ".env"))
if not loaded:
    raise ValueError(f".env not found in {os.getcwd()}")

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY missing in .env")

MODEL = "claude-haiku-4-5"   # use this first for cheap testing


config_list = [{
    "model": MODEL,
    "api_key": ANTHROPIC_API_KEY,
    "api_type": "anthropic",
}]

llm_config = {
    "config_list": config_list,
    "temperature": 0.4,
    "max_tokens": 500,
}

print("Model:", MODEL)
print("Key loaded:", bool(ANTHROPIC_API_KEY))
print("Key prefix:", ANTHROPIC_API_KEY[:10] if ANTHROPIC_API_KEY else "None")

Model: claude-haiku-4-5
Key loaded: True
Key prefix: sk-ant-api


## Define Scrum AI Agents

This setup maps Scrum roles to specialized AutoGen agents:
- Product Owner
- Scrum Master
- Business Analyst
- UI/UX Designer
- Developer
- QA Engineer
- DevOps Engineer

In [3]:
customer_proxy = UserProxyAgent(
    name="Customer_Proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
    default_auto_reply="",
)

product_owner = AssistantAgent(
    name="Product_Owner",
    system_message="""You are the Product Owner for a Scrum-based LMS project.

Responsibilities:
1. Define product vision and release goal for a GenAI-assisted personalized LMS.
2. Generate a prioritized Product Backlog with 20+ backlog items.
3. For each backlog item include: ID (PBI-001...), user story, acceptance criteria, business value, and priority.
4. Propose story points for each item and identify MVP scope.
5. Map backlog items to Phase 1 capabilities (personalization, AI tutor, instructor tools, analytics, AI project planning).

Output in well-structured markdown tables.""",
    llm_config=llm_config,
)

scrum_master = AssistantAgent(
    name="Scrum_Master",
    system_message="""You are the Scrum Master for this project.

Responsibilities:
1. Define Scrum cadence for 2-week sprints, ceremonies, and Definition of Done.
2. Create Sprint Goal(s) and a Sprint Plan for Sprint 1 and Sprint 2.
3. Build a Sprint Backlog for Sprint 1 from top-priority PBIs.
4. Provide daily scrum structure, impediment handling workflow, and risk mitigations.
5. Estimate team velocity and forecast sprint completion.

Output in structured markdown with clear sections and tables.""",
    llm_config=llm_config,
)

business_analyst = AssistantAgent(
    name="Business_Analyst",
    system_message="""You are a Business Analyst in a Scrum team.

Responsibilities:
1. Convert high-level needs into INVEST-compliant user stories.
2. Define measurable business KPIs and success criteria.
3. Provide requirement traceability: Epic -> Story -> Acceptance Criteria -> KPI.
4. Identify dependencies, assumptions, and constraints.
5. Highlight compliance/privacy considerations for LMS data.

Output in concise markdown tables.""",
    llm_config=llm_config,
)

uiux_designer = AssistantAgent(
    name="UI_UX_Designer",
    system_message="""You are the UI/UX Designer in the Scrum team.

Responsibilities:
1. Create UX backlog items (flows, wireframes, usability tests) tied to PBIs.
2. Define key user flows for students, instructors, and admins.
3. Provide design tasks for Sprint 1 and Sprint 2 with estimates.
4. Define accessibility requirements (WCAG-oriented) and usability acceptance criteria.
5. Specify handoff artifacts needed by developers and QA.

Output as markdown sections and tables.""",
    llm_config=llm_config,
)

developer = AssistantAgent(
    name="Developer",
    system_message="""You are the Developer in a Scrum team.

Responsibilities:
1. Break sprint stories into implementation tasks with IDs (DEV-T1...).
2. Include estimates (hours), dependencies, and technical approach.
3. Propose architecture increments for each sprint.
4. Provide API/data model tasks and AI integration tasks.
5. Define code quality gates (reviews, linting, testing).

Output in markdown tables and phased task lists.""",
    llm_config=llm_config,
)

qa_engineer = AssistantAgent(
    name="QA_Engineer",
    system_message="""You are the QA Engineer in a Scrum team.

Responsibilities:
1. Create a sprint-aligned test strategy (functional + non-functional + AI quality).
2. Define test cases with IDs (TC-001...) mapped to stories and acceptance criteria.
3. Add regression, performance, security, and AI response evaluation checks.
4. Provide Definition of Done quality criteria and release readiness checklist.
5. Estimate QA effort per sprint.

Output in markdown tables with traceability links.""",
    llm_config=llm_config,
)

devops_engineer = AssistantAgent(
    name="DevOps_Engineer",
    system_message="""You are the DevOps Engineer in a Scrum team.

Responsibilities:
1. Define CI/CD pipeline tasks for sprint delivery.
2. Provide environment strategy (dev, test, staging, prod) and deployment flow.
3. Add observability, alerting, rollback, and release automation tasks.
4. Include security and secrets management requirements.
5. Estimate DevOps workload and operational readiness milestones.

Output in concise markdown with sprint mapping.""",
    llm_config=llm_config,
)

print("Scrum agents created successfully.")
for name in [
    "Customer_Proxy",
    "Product_Owner",
    "Scrum_Master",
    "Business_Analyst",
    "UI_UX_Designer",
    "Developer",
    "QA_Engineer",
    "DevOps_Engineer",
]:
    print(f"  - {name}")


Scrum agents created successfully.
  - Customer_Proxy
  - Product_Owner
  - Scrum_Master
  - Business_Analyst
  - UI_UX_Designer
  - Developer
  - QA_Engineer
  - DevOps_Engineer


## Customer Problem Statement (Phase 2 Scrum)

The same LMS problem from Phase 1 is re-used, but the expected output is now Scrum artifacts and sprint planning.

In [4]:
customer_message = (
    "We are continuing our GenAI-Assisted Personalized LMS project from Phase 1, "
    "but now we need a Scrum-based project plan. Build agile artifacts for a team "
    "that includes Product Owner, Scrum Master, Business Analyst, UI/UX Designer, "
    "Developer, QA Engineer, and DevOps Engineer.\n\n"
    "Project Scope (same as Phase 1): personalized learning paths, AI tutor chatbot, "
    "AI-assisted instructor content generation, student/instructor analytics dashboards, "
    "and AI-assisted project planning support.\n\n"
    "Expected Phase 2 Output: product backlog with priorities, sprint goals, sprint backlog, "
    "story points, task breakdown, QA plan, CI/CD and release plan, risks, and velocity-based "
    "forecasting for at least two 2-week sprints. Include traceability and Definition of Done."
)

print("Phase 2 customer statement loaded.")
print(f"Message length: {len(customer_message)} characters")

Phase 2 customer statement loaded.
Message length: 749 characters


In [5]:
agents = [
    customer_proxy,
    product_owner,
    scrum_master,
    business_analyst,
    uiux_designer,
    developer,
    qa_engineer,
    devops_engineer,
]

groupchat = GroupChat(
    agents=agents,
    messages=[],
    max_round=8,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

print("GroupChat configured:")
print("  Speaker selection: round_robin")
print(f"  Max rounds: {groupchat.max_round}")
print(f"  Agents: {[a.name for a in agents]}")


GroupChat configured:
  Speaker selection: round_robin
  Max rounds: 8
  Agents: ['Customer_Proxy', 'Product_Owner', 'Scrum_Master', 'Business_Analyst', 'UI_UX_Designer', 'Developer', 'QA_Engineer', 'DevOps_Engineer']


## Run Scrum Experiment


In [7]:

groupchat.messages.clear()
print("Starting fresh Scrum GroupChat experiment...")

start_time = time.time()

chat_result = customer_proxy.initiate_chat(
    manager,
    message=customer_message,
)

total_elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(f"  Scrum GroupChat completed in {total_elapsed:.1f}s")
print("=" * 60)


Starting fresh Scrum GroupChat experiment...
Customer_Proxy (to chat_manager):

We are continuing our GenAI-Assisted Personalized LMS project from Phase 1, but now we need a Scrum-based project plan. Build agile artifacts for a team that includes Product Owner, Scrum Master, Business Analyst, UI/UX Designer, Developer, QA Engineer, and DevOps Engineer.

Project Scope (same as Phase 1): personalized learning paths, AI tutor chatbot, AI-assisted instructor content generation, student/instructor analytics dashboards, and AI-assisted project planning support.

Expected Phase 2 Output: product backlog with priorities, sprint goals, sprint backlog, story points, task breakdown, QA plan, CI/CD and release plan, risks, and velocity-based forecasting for at least two 2-week sprints. Include traceability and Definition of Done.

--------------------------------------------------------------------------------

Next speaker: Product_Owner

Product_Owner (to chat_manager):

# GenAI-Assisted Persona

In [8]:
output_dir = "Phase2_Experiment_Results_claude_haiku"
os.makedirs(output_dir, exist_ok=True)

agent_names = [
    "Product_Owner",
    "Scrum_Master",
    "Business_Analyst",
    "UI_UX_Designer",
    "Developer",
    "QA_Engineer",
    "DevOps_Engineer",
]

agent_system_prompts = {
    a.name: a.system_message
    for a in agents
    if hasattr(a, "system_message") and a.system_message
}

#  token estimate for Claude
def estimate_tokens(text):
    if not text:
        return 0
    return max(1, len(text) // 4)

all_results = []
for msg in groupchat.messages:
    name = msg.get("name", "")
    content = msg.get("content", "")
    if name in agent_names and content:
        output_tokens = estimate_tokens(content)
        sys_prompt = agent_system_prompts.get(name, "")
        input_tokens = estimate_tokens(sys_prompt) + estimate_tokens(customer_message)
        total_tokens_agent = input_tokens + output_tokens

        all_results.append({
            "agent": name,
            "response": content,
            "tokens": total_tokens_agent,
            "model": MODEL,
        })

total_tokens = sum(r["tokens"] for r in all_results)

for result in all_results:
    filepath = os.path.join(output_dir, f"{result['agent']}_output.md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {result['agent'].replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {result['model']}\n\n")
        f.write(f"**Estimated Tokens Used:** {result['tokens']}\n\n")
        f.write("---\n\n")
        f.write(result["response"])
    print(f"Saved: {filepath}")

combined_path = os.path.join(output_dir, "Phase2_Scrum_Complete_Report.md")
section_names = {
    "Product_Owner": "SECTION 2 - PRODUCT OWNER",
    "Scrum_Master": "SECTION 3 - SCRUM MASTER",
    "Business_Analyst": "SECTION 4 - BUSINESS ANALYST",
    "UI_UX_Designer": "SECTION 5 - UI/UX DESIGNER",
    "Developer": "SECTION 6 - DEVELOPER",
    "QA_Engineer": "SECTION 7 - QA ENGINEER",
    "DevOps_Engineer": "SECTION 8 - DEVOPS ENGINEER",
}

with open(combined_path, "w", encoding="utf-8") as f:
    f.write("# Phase 2 - Scrum Project Plan\n")
    f.write("# GenAI-Assisted Personalized Learning Management System (LMS)\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"**LLM Model:** {MODEL}\n\n")
    f.write("**Framework:** AutoGen (pyautogen) - GroupChat Multi-Agent Simulation\n\n")
    f.write("---\n\n")
    f.write("## SECTION 1 - CUSTOMER PROXY CONTEXT\n\n")
    f.write("### 1.1 Scrum Problem Statement\n\n")
    f.write(customer_message.strip())
    f.write("\n\n---\n\n")

    for result in all_results:
        agent = result["agent"]
        if agent in section_names:
            f.write(f"## {section_names[agent]}\n\n")
            f.write(f"**Model:** {result['model']} | **Estimated Tokens:** {result['tokens']:,}\n\n")
            f.write(result["response"])
            f.write("\n\n---\n\n")

print(f"Saved combined report: {combined_path}")

metadata = {
    "experiment": {
        "title": "Phase 2 - Scrum Project Planning",
        "project": "GenAI-Assisted Personalized LMS",
        "model": MODEL,
        "framework": f"AutoGen (pyautogen v{autogen.__version__}) - GroupChat",
        "speaker_selection": "round_robin",
        "date": datetime.now().isoformat(),
        "total_estimated_tokens": total_tokens,
        "total_time_seconds": round(total_elapsed, 2),
    },
    "agents": [
        {"name": r["agent"], "estimated_tokens": r["tokens"], "model": r["model"]}
        for r in all_results
    ],
    "groupchat": {
        "total_messages": len(groupchat.messages),
        "max_round": groupchat.max_round,
    },
}

metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata: {metadata_path}")

Saved: Phase2_Experiment_Results_claude_haiku/Product_Owner_output.md
Saved: Phase2_Experiment_Results_claude_haiku/Scrum_Master_output.md
Saved: Phase2_Experiment_Results_claude_haiku/Business_Analyst_output.md
Saved: Phase2_Experiment_Results_claude_haiku/UI_UX_Designer_output.md
Saved: Phase2_Experiment_Results_claude_haiku/Developer_output.md
Saved: Phase2_Experiment_Results_claude_haiku/QA_Engineer_output.md
Saved: Phase2_Experiment_Results_claude_haiku/DevOps_Engineer_output.md
Saved combined report: Phase2_Experiment_Results_claude_haiku/Phase2_Scrum_Complete_Report.md
Saved metadata: Phase2_Experiment_Results_claude_haiku/experiment_metadata.json


## Experiment Summary

In [9]:
print("=" * 60)
print("  SCRUM EXPERIMENT COMPLETE")
print("=" * 60)
print()
print(f"  Model:         {MODEL}")
print(f"  Framework:     AutoGen (pyautogen v{autogen.__version__})")
print("  Method:        Scrum (Agile)")
print("  GroupChat:     round_robin")
print(f"  Total Agents:  {len(all_results)}")
print(f"  Total Messages:{len(groupchat.messages)}")
print(f"  Total Tokens (Estimated):  {total_tokens:,}")
print(f"  Total Time:    {total_elapsed:.1f}s")
print()
print("  Agent Breakdown:")
for r in all_results:
    print(f"    {r['agent']:22s} | Est. Tokens: {r['tokens']:7,} | Response: {len(r['response']):6,} chars")
print()
print(f"  Output Directory: {output_dir}/")
print("=" * 60)

  SCRUM EXPERIMENT COMPLETE

  Model:         claude-haiku-4-5
  Framework:     AutoGen (pyautogen v0.3.2)
  Method:        Scrum (Agile)
  GroupChat:     round_robin
  Total Agents:  7
  Total Messages:8
  Total Tokens (Estimated):  5,407
  Total Time:    0.1s

  Agent Breakdown:
    Product_Owner          | Est. Tokens:     799 | Response:  1,887 chars
    Scrum_Master           | Est. Tokens:     773 | Response:  1,867 chars
    Business_Analyst       | Est. Tokens:     759 | Response:  1,875 chars
    UI_UX_Designer         | Est. Tokens:     770 | Response:  1,876 chars
    Developer              | Est. Tokens:     761 | Response:  1,885 chars
    QA_Engineer            | Est. Tokens:     778 | Response:  1,895 chars
    DevOps_Engineer        | Est. Tokens:     767 | Response:  1,885 chars

  Output Directory: Phase2_Experiment_Results_claude_haiku/
